<table style="width: 100%; border: none; background: #f8f9fa; padding: 20px; border-radius: 10px;">
    <tr>
        <td style="width: 22%; border: none; vertical-align: middle; text-align: center;">
            <img src="https://www.inf.ucv.cl/wp-content/uploads/2020/05/logo_escuela.jpg" width="170">
        </td>
        <td style="width: 56%; border: none; vertical-align: middle; text-align: center; padding: 0 20px;">
            <h1 style="font-size: 28px; color: #1a3b5c; margin-bottom: 16px; border-bottom: 2px solid #1a3b5c; padding-bottom: 8px;">Tarea 2: Clasificacion, Costos Asimetricos y Arboles</h1>
            <p style="font-size: 16px; margin: 8px 0;"><strong>Pontificia Universidad Catolica de Valparaiso</strong><br>
            <strong>Escuela de Ingenieria Informatica</strong></p>
            <p style="font-size: 14px; color: #333; margin: 12px 0;"><strong>OII432 Machine Learning</strong><br>
            <strong>Semestre: 2-2026</strong><br>
            <strong>Profesor: Carlos Valle</strong></p>
        </td>
        <td style="width: 22%; border: none; vertical-align: middle; text-align: center;">
            <img src="https://upload.wikimedia.org/wikipedia/commons/c/cb/PUCV_Escudo2016.svg" width="110">
        </td>
    </tr>
</table>

# Pregunta 2: desafío Kaggle
Integrantes:
- Nelson Fuentes
- Eduardo Pérez
- Eduardo Sandoval
- Ricardo Pastén

Se predice si una licitación quedará desierta con información disponible al publicarla. La pauta adjunta confirma que la métrica es F1 para la clase desierta.


El trabajo conserva seis partes: datos y protocolo, regresión logística, comparación de modelos, elección del umbral, árboles y ensambles, y decisión final.

Esta revisión agrega dos mejoras sobre la versión anterior: (1) **tasas históricas de desierta** por comprador, categoría, comprador × categoría y comprador × título (relicitaciones), calculadas solo con etiquetas ya maduras y suavizadas; y (2) un **modelo de boosting de gradiente** que se compara con las regresiones logísticas y los árboles bajo el mismo protocolo temporal. También se audita el plazo de postulación y la fecha de publicación antes de modelar.

Se conserva lo anterior: texto con unigramas y vocabulario acotado, selección en tres ventanas temporales y la comparación del **monto estimado en CLP**, sin mezclar monedas. F1 sigue siendo la métrica de la tarea; Average Precision (AP) permite seleccionar modelos antes de elegir el umbral.

Se mantiene la corrección del protocolo anterior: una publicación anterior al corte puede seguir abierta y su etiqueta no estaría disponible. El entrenamiento, las tasas históricas y las decisiones usan solo casos cuyo cierre más un margen de 30 días precede al corte. Ese margen es una aproximación conservadora, porque no tenemos la fecha real de publicación del desenlace.

## Librerías


In [66]:
import hashlib
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from IPython.display import display
from scipy.special import expit
from scipy.sparse import csr_matrix, hstack

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
TARGET = "y_desierta"
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_palette("deep")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

NumPy 1.26.2 | pandas 2.3.3 | scikit-learn 1.9.1


# 1. Datos y protocolo

Se predice `y_desierta`: **1** significa que la licitación quedó desierta y **0**, que no quedó desierta. Primero se revisan los archivos, los datos faltantes y la distribución de ambas clases. Después se separan los datos por fecha y se fija cuánto pesa cada tipo de error.


### Carga de datos

In [67]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
assert list(pd.read_csv("sample_submission.csv", nrows=0).columns) == ["id", TARGET]
# Solo se leen los identificadores de la plantilla; sus etiquetas de ejemplo se ignoran.
sample_submission_df = pd.read_csv("sample_submission.csv", usecols=["id"])

### Inspección e integridad

In [68]:
for nombre, df in {"train": train_df, "test": test_df}.items():
    print(nombre, df.shape)
    display(df.head(3))

train (152553, 24)


,id,pub_fecha,pub_titulo,pub_descripcion,pub_monto_estimado,pub_moneda,pub_metodo,pub_modalidad,pub_n_items,pub_cantidad_total,...,pub_cierre,pub_dias_postulacion,pub_tiene_consultas,pub_dias_consultas,comprador,comprador_id,comprador_rut,region,direccion,y_desierta
0,248328745624,2024-03-25T15:32:20Z,LP04-2024 AV. UNO SUR AYSEN,"CONSTRUCCIÓN CALZADA H.C.V. AVENIDA UNO SUR, P...",NaN,NaN,open,Licitación Pública MOP (O1),1,1.0,...,2024-04-12T10:00:00Z,17.769,False,11.0,SERVICIO DE VIVIENDA Y URBANIZACION XI REGION ...,CL-MP-1653,618230007,Región Aysén del General Carlos Ibáñez del Campo,RAMON FREIRE Nº 5,0
1,760555527084,2024-03-21T14:29:28Z,LICITACION PUBLICA PARA OBRAS DE INFRAESTRUCTU...,licitar las obras de infraestructura de mejora...,NaN,NaN,open,Licitación Pública entre a 2000 y 5000 UTM (LQ),1,1.0,...,2024-04-15T20:00:00Z,25.230,False,21.0,CORPORACION MUNICIPAL DE EDUCACION Y SALUD DE ...,CL-MP-1274286,709255002,Región Metropolitana de Santiago,O'HIGGINS 840,1
2,224829110991,2024-01-26T12:37:41Z,ADQUISICION DE INSUMOS DE ENFERMERÍA PARA DSM,ADQUISICION DE INSUMOS DE ENFERMERÍA PARA USUA...,3500000.0,CLP,open,Licitación Pública Menor a 100 UTM (L1),55,12150.0,...,2024-02-01T13:24:00Z,6.032,False,3.0,I MUNICIPALIDAD DE CHANCO | I MUNICIPALIDAD DE...,CL-MP-4391,69120300K,Región del Maule,Ignacio Carrera Pinto Nº 31,0


test (21829, 23)


,id,pub_fecha,pub_titulo,pub_descripcion,pub_monto_estimado,pub_moneda,pub_metodo,pub_modalidad,pub_n_items,pub_cantidad_total,...,pub_inicio,pub_cierre,pub_dias_postulacion,pub_tiene_consultas,pub_dias_consultas,comprador,comprador_id,comprador_rut,region,direccion
0,47254358932,2025-03-12T15:46:53Z,SC 36914 (CM) ADQUISICIÓN DE MOBILIARIO PARA C...,ADQUISICIÓN DE MOBILIARIO PARA CLÍNICA JURÍDIC...,4999999.0,CLP,open,Licitación Pública Menor a 100 UTM (L1),3,24.0,...,2025-03-12T15:46:47Z,2025-03-20T15:30:00Z,7.988,False,2.0,UNIVERSIDAD DE PLAYA ANCHA DE CIENCIAS DE LA E...,CL-MP-5385,707547006,Región Metropolitana de Santiago,Av. A Bello 2687
1,295671198916,2025-01-06T17:34:25Z,COMPRA DE MEDICAMENTOS PARA DEPTO SALUD MAULE,COMPRA DE MEDICAMENTOS PARA DEPTO SALUD MAULE,20000000.0,CLP,open,Licitación Pública Entre 100 y 1000 UTM (LE),45,123593.0,...,2025-01-06T17:34:19Z,2025-01-16T19:53:00Z,10.096,False,3.0,I MUNICIPALIDAD DE MAULE | I MUNICIPALIDAD DE ...,CL-MP-5605,691109003,Región del Maule,Balmaceda N° 350
2,214756376398,2025-02-26T12:48:51Z,ADQUISICIÓN DE BOLSAS CORPORATIVAS REUTILIZABL...,ADQUISICIÓN DE BOLSAS CORPORATIVAS REUTILIZABL...,NaN,NaN,open,Licitación Pública Entre 100 y 1000 UTM (LE),1,19000.0,...,2025-02-26T12:48:41Z,2025-03-03T15:00:00Z,5.091,False,2.0,I MUNICIPALIDAD DE QUILICURA | DEPARTAMENTO CO...,CL-MP-3475,690713004,Región Metropolitana de Santiago,NaN


In [69]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152553 entries, 0 to 152552
Data columns (total 24 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    152553 non-null  int64  
 1   pub_fecha             152553 non-null  object 
 2   pub_titulo            152544 non-null  object 
 3   pub_descripcion       152544 non-null  object 
 4   pub_monto_estimado    85287 non-null   float64
 5   pub_moneda            85251 non-null   object 
 6   pub_metodo            152485 non-null  object 
 7   pub_modalidad         152553 non-null  object 
 8   pub_n_items           152553 non-null  int64  
 9   pub_cantidad_total    152553 non-null  float64
 10  pub_categoria         152528 non-null  object 
 11  pub_categoria_id      152528 non-null  float64
 12  pub_n_categorias      152553 non-null  int64  
 13  pub_inicio            152553 non-null  object 
 14  pub_cierre            152526 non-null  object 
 15  

In [70]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21829 entries, 0 to 21828
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    21829 non-null  int64  
 1   pub_fecha             21829 non-null  object 
 2   pub_titulo            21829 non-null  object 
 3   pub_descripcion       21829 non-null  object 
 4   pub_monto_estimado    10230 non-null  float64
 5   pub_moneda            10226 non-null  object 
 6   pub_metodo            21734 non-null  object 
 7   pub_modalidad         21829 non-null  object 
 8   pub_n_items           21829 non-null  int64  
 9   pub_cantidad_total    21829 non-null  float64
 10  pub_categoria         21828 non-null  object 
 11  pub_categoria_id      21828 non-null  float64
 12  pub_n_categorias      21829 non-null  int64  
 13  pub_inicio            21829 non-null  object 
 14  pub_cierre            21823 non-null  object 
 15  pub_dias_postulacio

In [71]:
assert TARGET in train_df and TARGET not in test_df
assert train_df[TARGET].notna().all()
assert set(train_df[TARGET].unique()) == {0, 1}
assert set(train_df.columns) - {TARGET} == set(test_df.columns)
assert len(train_df) == 152553 and len(test_df) == 21829
for df in (train_df, test_df, sample_submission_df):
    assert df["id"].notna().all() and df["id"].is_unique
assert set(train_df["id"]).isdisjoint(test_df["id"])
assert sample_submission_df["id"].equals(test_df["id"])
assert TARGET not in sample_submission_df

### Distribución de la clase objetivo

In [72]:
frecuencias = train_df[TARGET].value_counts().sort_index()
pd.DataFrame({"registros": frecuencias, "porcentaje": 100 * frecuencias / len(train_df)})

,registros,porcentaje
y_desierta,,
0,122006,79.976139
1,30547,20.023861


Cerca del 20% de las licitaciones de 2024 quedaron desiertas. Predecir siempre «no desierta» tendría alta exactitud, pero no detectaría ninguna desierta. Esta distribución global es descriptiva: los pesos de los errores se calculan solo con las etiquetas admisibles del primer entrenamiento. Ninguna decisión usa etiquetas de 2025.

### Valores nulos

In [73]:
nulos = train_df.isna().sum()
pd.DataFrame({"nulos": nulos, "porcentaje": 100 * nulos / len(train_df)})

,nulos,porcentaje
id,0,0.000000
pub_fecha,0,0.000000
pub_titulo,9,0.005900
pub_descripcion,9,0.005900
pub_monto_estimado,67266,44.093528
pub_moneda,67302,44.117127
pub_metodo,68,0.044575
pub_modalidad,0,0.000000
pub_n_items,0,0.000000
pub_cantidad_total,0,0.000000


`pub_monto_estimado` y `pub_moneda` tienen cerca del 44% de valores nulos.

### Ausencia de monto y moneda

In [74]:
pd.crosstab(
    train_df["pub_monto_estimado"].isna(),
    train_df["pub_moneda"].isna(),
    rownames=["Monto nulo"],
    colnames=["Moneda nula"]
)

Moneda nula,False,True
Monto nulo,,
False,85251,36
True,0,67266


### Relación entre monto nulo y variable objetivo

Se compara la proporción de licitaciones desiertas (`y_desierta = 1`) entre las que tienen `pub_monto_estimado` informado y las que lo tienen nulo. La ausencia se identifica con `.isna()`; un monto igual a cero se considera informado.

La tabla muestra los conteos de ambas clases y la **tasa de desiertas dentro de cada grupo**: desiertas / total del grupo × 100. Comparar estas tasas permite considerar que los grupos tienen tamaños diferentes. El gráfico incluye la tasa global como referencia y se calcula la diferencia **monto nulo − monto informado**, expresada en puntos porcentuales: un valor positivo indica una mayor proporción de desiertas entre los casos con monto nulo.

En estos datos, la tasa es aproximadamente **20,023% con monto informado** y **20,025% con monto nulo**: una diferencia de apenas **0,002 puntos porcentuales**. Por sí sola, la ausencia del monto prácticamente no distingue las clases en esta muestra.

El análisis es descriptivo y usa solo `train_df`. El aporte de una indicadora de monto faltante, incluidas posibles interacciones con modalidad o comprador, debe comprobarse mediante validación temporal.

In [75]:
# Cruce entre ausencia del monto y desenlace, sin modificar train_df.
relacion_monto_objetivo = pd.crosstab(
    train_df["pub_monto_estimado"].isna(), train_df[TARGET]
).reindex(index=[False, True], columns=[0, 1], fill_value=0)
relacion_monto_objetivo.index = pd.Index(
    ["Monto informado", "Monto nulo"], name="Estado del monto"
)
relacion_monto_objetivo.columns = ["y = No desierta", "y = desierta"]
relacion_monto_objetivo["Registros"] = relacion_monto_objetivo.sum(axis=1)
display(relacion_monto_objetivo)

tasa_global_desierta = 100 * train_df[TARGET].mean()

,y = No desierta,y = desierta,Registros
Estado del monto,,,
Monto informado,68210,17077,85287
Monto nulo,53796,13470,67266


In [76]:
# Se eliminan las columnas de comprador y RUT del comprador por ser redundantes.
columnas_a_eliminar = ["comprador", "comprador_rut"]
train_df = train_df.drop(columns=columnas_a_eliminar, errors="ignore")

In [77]:
# Se elimina la columna de categoría por ser redundante.
columnas_a_eliminar = ["pub_categoria"]
train_df = train_df.drop(columns=columnas_a_eliminar, errors="ignore")

In [78]:
# Se elimina la columna de categoría por ser insignificante.
columnas_a_eliminar = ["direccion"]
train_df = train_df.drop(columns=columnas_a_eliminar, errors="ignore")

In [79]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152553 entries, 0 to 152552
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    152553 non-null  int64  
 1   pub_fecha             152553 non-null  object 
 2   pub_titulo            152544 non-null  object 
 3   pub_descripcion       152544 non-null  object 
 4   pub_monto_estimado    85287 non-null   float64
 5   pub_moneda            85251 non-null   object 
 6   pub_metodo            152485 non-null  object 
 7   pub_modalidad         152553 non-null  object 
 8   pub_n_items           152553 non-null  int64  
 9   pub_cantidad_total    152553 non-null  float64
 10  pub_categoria_id      152528 non-null  float64
 11  pub_n_categorias      152553 non-null  int64  
 12  pub_inicio            152553 non-null  object 
 13  pub_cierre            152526 non-null  object 
 14  pub_dias_postulacion  152530 non-null  float64
 15  

### Sextiles del monto sin valores atípicos

Usamos montos en **CLP** y excluimos los valores fuera de **[Q1 − 1,5 × RIQ, Q3 + 1,5 × RIQ]**, donde RIQ = Q3 − Q1. Luego dividimos los montos restantes en **sextiles**: seis grupos con aproximadamente la misma cantidad de registros. La tabla muestra los conteos de no desiertas y desiertas, y el porcentaje de desiertas por grupo. Los montos nulos se cuentan aparte. El filtro se aplica solo a este análisis.

In [ ]:
montos = train_df.loc[
    train_df["pub_moneda"].eq("CLP") | train_df["pub_monto_estimado"].isna(),
    ["pub_monto_estimado", TARGET]
].copy()
q1, q3 = montos["pub_monto_estimado"].quantile([0.25, 0.75])
riq = q3 - q1
conservar = (
    montos["pub_monto_estimado"].between(q1 - 1.5 * riq, q3 + 1.5 * riq)
    | montos["pub_monto_estimado"].isna()
)
print(f"Montos atípicos excluidos: {(~conservar).sum()}")
montos = montos.loc[conservar].copy()
montos["Intervalo"] = pd.qcut(montos["pub_monto_estimado"], q=6)
montos["Intervalo"] = montos["Intervalo"].cat.add_categories("Monto nulo").fillna("Monto nulo")
display(montos.groupby("Intervalo", observed=False).agg(
    Registros=(TARGET, "size"),
    Montos_nulos=("pub_monto_estimado", lambda x: x.isna().sum()),
    No_desiertas=(TARGET, lambda y: y.eq(0).sum()),
    Desiertas=(TARGET, "sum"),
    Desiertas_pct=(TARGET, lambda y: 100 * y.mean())
).round(2))

### Distribución de los montos

El histograma muestra los montos en millones de CLP, usando el filtro de atípicos anterior. Las líneas punteadas indican los límites entre sextiles. Los montos nulos se omiten del gráfico.

In [ ]:
montos_grafico = montos["pub_monto_estimado"].dropna() / 1_000_000
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(montos_grafico, bins=40, color="steelblue", edgecolor="white")
for i, limite in enumerate(montos_grafico.quantile(np.arange(1, 6) / 6)):
    ax.axvline(limite, color="darkorange", linestyle="--",
               label="Límites de sextiles" if i == 0 else None)
ax.set(title="Distribución de montos sin atípicos",
       xlabel="Monto (millones de CLP)", ylabel="Cantidad de licitaciones")
ax.legend()
plt.tight_layout()
plt.show()